# Top-N expected loss share over time

Pick an index, two dates and N. The **top-N share** is the expected loss of the N widest names (largest `all_upfront`, i.e. present value of expected loss) times their weights, divided by the index's
total expected loss (weight x `all_upfront` summed over all priced names). The chart shows it for every date between the two, against a benchmark: the share the top N would hold if every live name
carried the same expected loss (N divided by the number of live names), so the gap between the lines is how concentrated the index is.

- Names that defaulted out of a series stay in it, at their original weight, until their index ex-date (`teams.structured_credit_abhutra.credit_events`).
- A name with no price on a date is left out of that date. Shaded bands are dates where the priced names carry less than `SPARSE_BELOW` of the live index weight.
- Below the chart: the two dates side by side, and which names are in the top N on each (who entered, who left).

In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'analysis' / 'cdx_spreads.py').exists())     # project root, wherever the notebook is opened from
sys.path.insert(0, str(ROOT / 'analysis'))

import pandas as pd
import cdx_spreads as cs
import index_metrics as im
import charts
from IPython.display import HTML, display

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)

show = cs.show_table      # no-wrap HTML table with fixed decimals
print(', '.join(cs.load_index_list()['index_name']))
cs.print_data_as_of()          # which published snapshot (data/) this notebook is reading


IG47, IG46, IG45, IG44, IG43, IG42, IG41, IG40, IG39, IG38, HY47, HY46, HY45, HY44, HY43, HY42, HY41, HY40, HY39, HY38, XO46, XO45, XO44, XO43, XO42, XO41, XO40, XO39, XO38, XO37, MA46, MA45, MA44, MA43, MA42, MA41, MA40, MA39, MA38, MA37
data published 2026-10-26 08:34; prices through 2026-10-23


## Parameters

In [2]:
INDEX = 'HY42'
DATE_X = '2025-03-31'     # start; snaps to the last available date on or before
DATE_Y = '2026-10-23'     # end;   same
N = 10                    # how many of the widest names
SPARSE_BELOW = 0.95       # shade dates where the priced names carry less than this share of the live index weight

## Top-N share over time

In [3]:
def credit_events(index_name, first, last):
    """Names that defaulted out of the series inside the plotted window, as (ex-date, label) pairs for the chart markers."""
    if not im.EVENTS_FILE.exists():
        return []
    ev = pd.read_parquet(im.EVENTS_FILE)
    ev = ev[(ev['index_name'] == index_name) & (ev['index_ex_date'] >= first) & (ev['index_ex_date'] <= last)]
    return [(r.index_ex_date, f'{r.sn_ticker} left {index_name} on {r.index_ex_date:%d %b %Y} (credit event)') for r in ev.itertuples()]


el = im.name_el(INDEX)
met = im.index_metrics(INDEX, top_ns=(N,), el=el).set_index('published_date')
avail = met.index.to_series()
dx, dy = cs._asof(avail, DATE_X), cs._asof(avail, DATE_Y)
win = met.loc[dx:dy]
col = f'top{N}_share'
share = win[col] * 100
bench = N / win['n_members'] * 100          # share the top N would hold if every live name had the same expected loss
dates = list(win.index)
bands = charts.runs(dates, (win['priced_weight'] < SPARSE_BELOW).values)
tip = [f"index expected loss {r.index_el:.2f} pts · widest name {r.max_el:.1f} pts · {int(r.n_priced)} of {int(r.n_members)} names priced" for r in win.itertuples()]
tab = {'index EL (pts)': [f'{v:.2f}' for v in win['index_el']], 'priced names': [f'{int(a)} / {int(b)}' for a, b in zip(win['n_priced'], win['n_members'])]}
events = credit_events(INDEX, dx, dy)

print(f"{INDEX}: top {N} share {dx:%Y-%m-%d} {share.iloc[0]:.1f}% -> {dy:%Y-%m-%d} {share.iloc[-1]:.1f}% (benchmark {bench.iloc[0]:.1f}% -> {bench.iloc[-1]:.1f}%) | "
      f"{(win['priced_weight'] < SPARSE_BELOW).sum()} of {len(win)} dates sparse, {len(events)} credit events in the window")
display(HTML(charts.line_chart(
    dates, [dict(name=f'Top {N} share of expected loss', short=f'Top {N}', values=share.values), dict(name=f'Benchmark: {N} / live names', short='Benchmark', values=bench.values, muted=True)],
    title=f'{INDEX} share of index expected loss held by the {N} widest names',
    subtitle='Weighted expected loss of the N widest names divided by the index total. The gray line is the share if every name carried the same expected loss.',
    y_label='% of index expected loss', fmt=lambda v: f'{v:.1f}%', bands=bands, band_label=f'Sparse pricing (priced weight < {SPARSE_BELOW:.0%})' if len(bands) else '',
    events=events, event_label='Credit event' if events else '', tip_extra=tip, table_extra=tab)))

HY42: top 10 share 2025-03-31 37.2% -> 2026-10-23 63.4% (benchmark 10.0% -> 10.2%) | 0 of 410 dates sparse, 2 credit events in the window


## The two dates

In [4]:
def row(d):
    r = met.loc[d]
    w = im.ranked(INDEX, d, el).iloc[0]
    return {'date': f'{d:%Y-%m-%d}', f'top {N} share (%)': r[col] * 100, 'benchmark (%)': N / r['n_members'] * 100, 'concentration (x benchmark)': r[col] / (N / r['n_members']),
            'index EL (pts)': r['index_el'], 'dispersion': r['dispersion'], 'widest name': f"{w['member_name']} ({w['all_upfront']:.1f})",
            'live names': int(r['n_members']), 'priced names': int(r['n_priced'])}
two = pd.DataFrame([row(dx), row(dy)])
chg = {'date': 'change'}
for c in (f'top {N} share (%)', 'benchmark (%)', 'concentration (x benchmark)', 'index EL (pts)', 'dispersion'):
    chg[c] = two[c].iloc[1] - two[c].iloc[0]
two = pd.concat([two, pd.DataFrame([chg])], ignore_index=True)
show(two, numeric={f'top {N} share (%)': 1, 'benchmark (%)': 1, 'concentration (x benchmark)': 2, 'index EL (pts)': 2, 'dispersion': 2})

date,top 10 share (%),benchmark (%),concentration (x benchmark),index EL (pts),dispersion,widest name,live names,priced names
2025-03-31,37.2,10.0,3.72,12.52,13.28,iHeartCommunications Inc (63.9),100.0,100.0
2026-10-23,63.4,10.2,6.21,6.53,13.64,CSC Holdings LLC (79.4),98.0,97.0
change,26.1,0.2,2.49,-5.99,0.36,,,


## Who is in the top N
The names in the top N on either date, with rank and `all_upfront` on each, and their share of the index's expected loss. Rank is among all priced names (1 = widest).

In [5]:
rx, ry = im.ranked(INDEX, dx, el), im.ranked(INDEX, dy, el)
a = rx[rx['rank'] <= N].set_index('market_cds_ticker')
b = ry[ry['rank'] <= N].set_index('market_cds_ticker')
tickers = list(dict.fromkeys(list(b.index) + [t for t in a.index if t not in b.index]))
rows = []
for t in tickers:
    x = rx[rx['market_cds_ticker'] == t]
    y = ry[ry['market_cds_ticker'] == t]
    name = (y if len(y) else x)['member_name'].iloc[0]
    inx, iny = t in a.index, t in b.index
    rows.append({'name': name, 'ticker': t, 'status': 'in top N on both dates' if inx and iny else ('entered' if iny else 'left'),
                 'rank X': x['rank'].iloc[0] if len(x) else pd.NA, 'rank Y': y['rank'].iloc[0] if len(y) else pd.NA,
                 f'all_upfront {dx:%Y-%m-%d}': x['all_upfront'].iloc[0] if len(x) else float('nan'), f'all_upfront {dy:%Y-%m-%d}': y['all_upfront'].iloc[0] if len(y) else float('nan'),
                 'share of EL X (%)': x['share'].iloc[0] * 100 if len(x) else float('nan'), 'share of EL Y (%)': y['share'].iloc[0] * 100 if len(y) else float('nan')})
names = pd.DataFrame(rows).sort_values(['rank Y', 'rank X'], na_position='last').reset_index(drop=True)
names['rank X'] = names['rank X'].astype('Int64')
names['rank Y'] = names['rank Y'].astype('Int64')
show(names, numeric={f'all_upfront {dx:%Y-%m-%d}': 1, f'all_upfront {dy:%Y-%m-%d}': 1, 'share of EL X (%)': 1, 'share of EL Y (%)': 1})

name,ticker,status,rank X,rank Y,all_upfront 2025-03-31,all_upfront 2026-10-23,share of EL X (%),share of EL Y (%)
CSC Holdings LLC,CSCHLD,in top N on both dates,7,1,43.1,79.4,3.4,12.4
Hertz Corp/The,HTZ,in top N on both dates,2,2,63.5,68.1,5.1,10.6
Xerox Corp,XRX,in top N on both dates,8,3,39.6,58.6,3.2,9.2
iHeartCommunications Inc,IHRT,in top N on both dates,1,4,63.9,53.4,5.1,8.3
Staples Inc,SPLS,in top N on both dates,4,5,48.3,35.3,3.9,5.5
Domtar Corp,UFS,entered,19,6,19.0,35.2,1.5,5.5
MPT Operating Partnership LP,MPW,entered,12,7,29.3,26.9,2.3,4.2
Sabre Holdings Corp,SABHLD,in top N on both dates,9,8,32.2,19.9,2.6,3.1
Bausch Health Cos Inc,BHCCN,in top N on both dates,5,9,47.9,17.9,3.8,2.8
Community Health Systems Inc,CYH,in top N on both dates,3,10,49.2,10.7,3.9,1.7
